# Session 07: 미니 프로젝트

## 목표

Module 1에서 배운 모든 것을 종합하여, 두 서비스를 **개발 → 실행 → 테스트**까지 합니다.

### 과제 목록

| 과제 | 서비스 | 내용 |
|------|--------|------|
| 과제 1 | 서비스 A | `/model/info` 엔드포인트 추가 |
| 과제 2 | 서비스 A | `/predict/batch` 엔드포인트 추가 |
| 과제 3 | 서비스 B | `/health`에 Gemini 연결 상태 체크 추가 |
| 과제 4 | 서비스 B | `/analyze/batch` 엔드포인트 추가 |

### 진행 방식
1. 각 과제의 설명을 읽고 코드를 작성합니다
2. 서버를 실행하고 테스트합니다
3. 체크리스트를 통해 완성도를 확인합니다

## 과제 1: 서비스 A - `/model/info` 엔드포인트 추가

### 요구사항
- **GET** `/model/info` 엔드포인트를 추가합니다
- 응답에 포함할 정보:
  - `model_name`: 모델 이름
  - `model_version`: 모델 버전
  - `features`: 사용하는 피처 목록
  - `threshold`: 승인 판정 임계값

### WHY 이 기능이 필요한가?
- 운영 중에 "지금 어떤 모델이 서빙되고 있는지" 확인할 수 있어야 합니다
- CI/CD 파이프라인에서 배포 후 검증 단계에 활용합니다

In [ ]:
# 여기에 코드를 작성하세요


## 과제 2: 서비스 A - `/predict/batch` 엔드포인트 추가

### 요구사항
- **POST** `/predict/batch` 엔드포인트를 추가합니다
- 요청: `List[LoanRequest]` (최대 100건)
- 응답: `List[LoanResponse]`

### WHY 배치 API가 필요한가?
- 여러 건을 한 번에 보내면 HTTP 오버헤드를 줄일 수 있습니다
- 실무에서 심사 시스템이 한 번에 수십~수백 건을 요청하는 경우가 많습니다

In [ ]:
# 여기에 코드를 작성하세요


## 과제 3: 서비스 B - `/health`에 Gemini 연결 상태 체크 추가

### 요구사항
- 기존 `/health` 엔드포인트를 개선합니다
- Gemini API에 간단한 ping을 보내 연결 상태를 확인합니다
- 응답에 `gemini_connected: bool` 필드를 추가합니다

### WHY Gemini 연결 체크가 필요한가?
- 서비스 B는 외부 API(Gemini)에 의존합니다
- Gemini API가 다운되면 서비스 B도 동작하지 않습니다
- health check에서 이를 감지하면 로드밸런서가 트래픽을 다른 인스턴스로 전환할 수 있습니다

In [ ]:
# 여기에 코드를 작성하세요


## 과제 4: 서비스 B - `/analyze/batch` 엔드포인트 추가

### 요구사항
- **POST** `/analyze/batch` 엔드포인트를 추가합니다
- 요청: `BatchReviewRequest` (최대 10건)
- 응답: `BatchReviewResponse` (results + total_analyzed)

### WHY 최대 10건인가? (서비스 A는 100건)
- 서비스 A: 로컬 ML 모델 추론 → 건당 수 ms → 100건도 빠름
- 서비스 B: 외부 LLM API 호출 → 건당 수백 ms~수 s → 10건이면 수 초~수십 초
- 너무 많으면 HTTP 타임아웃 위험

In [ ]:
# 여기에 코드를 작성하세요


## 체크리스트

모든 과제를 완료했는지 확인합니다.

### 서비스 A (대출 심사)

| 항목 | 확인 |
|------|------|
| `schemas.py` - LoanRequest(영어 필드명), LoanResponse 정의 | [ ] |
| `schemas.py` - BatchLoanRequest, BatchLoanResponse 추가 | [ ] |
| `schemas.py` - ModelInfoResponse 추가 | [ ] |
| `model.py` - LoanModel 클래스 (__init__ + load, FIELD_TO_COLUMN 매핑) | [ ] |
| `main.py` - lifespan으로 모델 로딩 (app.state 패턴) | [ ] |
| `main.py` - GET `/health` 엔드포인트 | [ ] |
| `main.py` - POST `/predict` 엔드포인트 | [ ] |
| `main.py` - POST `/predict/batch` 엔드포인트 | [ ] |
| `main.py` - GET `/model/info` 엔드포인트 | [ ] |
| `uvicorn app.main:app --port 8000`으로 실행 확인 | [ ] |

### 서비스 B (리뷰 분석)

| 항목 | 확인 |
|------|------|
| `schemas.py` - ReviewRequest, ReviewResponse 정의 | [ ] |
| `schemas.py` - BatchReviewRequest, BatchReviewResponse 추가 | [ ] |
| `gemini_client.py` - ReviewAnalyzer 클래스 (동기 def, 직접 GoogleAPIError 임포트) | [ ] |
| `main.py` - lifespan으로 Gemini 클라이언트 초기화 (app.state 패턴) | [ ] |
| `main.py` - GET `/health` (동기 def, Gemini 연결 체크 포함) | [ ] |
| `main.py` - POST `/analyze` (동기 def) 엔드포인트 | [ ] |
| `main.py` - POST `/analyze/batch` (동기 def) 엔드포인트 | [ ] |
| `uvicorn app.main:app --port 8000`으로 실행 확인 | [ ] |



### 완성한 서비스 구조

```
service-a-loan/           # ML 모델 기반 대출 심사
├── app/
│   ├── main.py           # FastAPI 앱 + 엔드포인트
│   ├── model.py          # 모델 로딩 + 추론
│   └── schemas.py        # Pydantic 스키마
└── models/               # 학습된 모델 파일

service-b-review/         # LLM 기반 리뷰 분석
├── app/
│   ├── main.py           # FastAPI 앱 + 엔드포인트
│   ├── gemini_client.py  # Gemini API 클라이언트
│   └── schemas.py        # Pydantic 스키마
```
